# Введение и цель этапа

Цель данного ноутбука — реализовать расчёт страхового запаса (`safety stock`) и на его основе получить рекомендацию по объёму заказа для каждого товара.

На предыдущих этапах были выполнены:
- очистка и агрегация помесячных продаж;
- XYZ-анализ ассортимента;
- baseline-прогноз спроса для товаров классов X и Y.

На этом этапе прогноз превращается в управленческое решение: сколько товара необходимо заказать с учётом риска нестабильного спроса.


## Входные данные и ограничения

В расчётах используются следующие данные:
- помесячные продажи (`monthly_sales`);
- baseline-прогноз спроса (`baseline_forecast`);
- XYZ-классификация товаров (`xyz_operational`);
- текущие остатки на складе (`stock_df`).

Ограничения:
- автоматический расчёт выполняется только для товаров классов X и Y;
- для товаров Z и `Unknown` прогноз не используется, решение принимается вручную;
- данные агрегированы по месяцам, без учёта `lead time` поставки.


## Логика расчёта `safety stock`

### Почему нужен `safety stock`

Прогноз спроса отражает ожидаемое среднее значение, но не учитывает колебания спроса.
`Safety stock` используется как защитный механизм от ошибок прогноза и нестабильности спроса.


### 3.1 Основа расчёта: стандартное отклонение

В качестве меры нестабильности используется стандартное отклонение месячных продаж:
- оно уже применялось при XYZ-анализе;
- отражает фактическую вариативность спроса;
- не требует предположений о распределении.

Для каждого SKU рассчитывается:
- среднее значение продаж;
- стандартное отклонение продаж.


In [14]:
import pandas as pd
from pathlib import Path

data_path = Path.cwd().parent.parent / "data" / "clean"
sales_data_path = data_path / "sales_data.csv"
xyz_operational_path = data_path / "xyz_operational_result.csv"
stock_path = data_path / "current_stocks_data.csv"


sales_data = pd.read_csv(sales_data_path)
xyz_operational = pd.read_csv(xyz_operational_path)
stocks = pd.read_csv(stock_path)

sales_data.tail()

,product,sku,qty,unit,month
6055,"Мороженое КОРОВКА ИЗ КОРЕНОВКИ ""Пломбир"" шокол...",NaN,7.0,шт,2025-10
6056,Эскимо Коровка из Кореновки пломбир МОЛ ШОК Бе...,NaN,4.0,шт,2025-10
6057,Рожок КОРОВКА ИЗ КОРЕНОВКИ пломбир с брусничны...,NaN,2.0,шт,2025-10
6058,"Мороженое КОРОВКА ИЗ КОРЕНОВКИ ""Эскимо"" в шоко...",NaN,1.0,шт,2025-10
6059,Подарочный сертификат 500 АЛЕНКА,NaN,1.0,шт,2025-10


### 3.2 Коэффициенты `safety stock` по XYZ-классам

Для учёта различной степени риска используются коэффициенты, зависящие от XYZ-класса:
- X (стабильные товары) — меньший коэффициент;
- Y (умеренно нестабильные) — больший коэффициент;
- Z / `Unknown` — `safety stock` не рассчитывается.

Формула:

`safe\_stock = k × std\_qty`

Где:
- `std_qty` — стандартное отклонение продаж;
- `k` — коэффициент, отражающий допустимый уровень риска.


In [15]:
k_by_xyz = {
    "Y - Средний": 0.5,
    "Y - Средний": 1.0,
    "Z - Нестабильный": 0.0,
    "Unknown": 0.0,
}

# 1) Стандартное отклонение продаж по каждому SKU
std_data = (
    sales_data.groupby("sku")["qty"]
    .std()
    .reset_index(name="std_qty")
)
std_data["std_qty"] = std_data["std_qty"].fillna(0).clip(lower=0)
result = std_data.merge(
    xyz_operational[["Артикул", "Класс товара"]],
    left_on="sku",
    right_on="Артикул",
    how="left"
)
result = result.drop(columns="Артикул")
result = result.rename(columns={"Класс товара": "Product_class"})

def calc_stock(row):
    if row["std_qty"] == 0:
        return 0
    k = k_by_xyz.get(row["Product_class"], 0)
    return row["std_qty"] * k
        
result["safety_stock"] = result.apply(calc_stock, axis=1)

result.head()




,sku,std_qty,Product_class,safety_stock
0,0104,0.000000,Unknown,0.000000
1,15033,2.869379,Y - Средний,2.869379
2,38010,3.003029,Z - Нестабильный,0.000000
3,38015,4.272534,Y - Средний,4.272534
4,38030,1.763834,Y - Средний,1.763834


## Расчёт рекомендуемого объёма заказа

### 4.1 Базовая формула

Рекомендуемый объём заказа рассчитывается как:

`order\_qty = forecast\_qty - stock\_qty + safe\_stock`

Где:
- `forecast_qty` — прогноз спроса на следующий месяц;
- `stock_qty` — текущий остаток;
- `safe_stock` — страховой запас.

### 4.2 Ограничения на результат

Для практического использования применяются дополнительные правила:
- если `order_qty < 0`, то `order_qty = 0`;
- итоговый заказ не может быть отрицательным;
- расчёт выполняется только для товаров классов X и Y.


In [16]:
def calculate_order_qty(
    sales_data: pd.DataFrame,
    xyz_operational: pd.DataFrame,
    safety_stock_df: pd.DataFrame | None = None,
    stock_df: pd.DataFrame | None = None,
) -> pd.DataFrame:
    """Расчет forecast_qty и order_qty по правилам XYZ.

    X - Стабильный  -> MA6
    Y - Средний     -> MA3
    """
    required_sales_cols = {"sku", "qty", "month"}
    missing_sales = required_sales_cols - set(sales_data.columns)
    if missing_sales:
        raise ValueError(f"В sales_data нет колонок: {sorted(missing_sales)}")

    required_xyz_cols = {"Артикул", "Класс товара"}
    missing_xyz = required_xyz_cols - set(xyz_operational.columns)
    if missing_xyz:
        raise ValueError(f"В xyz_operational нет колонок: {sorted(missing_xyz)}")

    window_by_class = {
        "X - Стабильный": 6,
        "Y - Средний": 3,
    }

    base = sales_data.copy()
    base["month"] = pd.to_datetime(base["month"])

    class_map = xyz_operational[["Артикул", "Класс товара"]].rename(
        columns={"Артикул": "sku", "Класс товара": "Product_class"}
    )
    base = base.merge(class_map, on="sku", how="left")
    base = base[base["Product_class"].isin(window_by_class)].copy()

    forecast_rows = []
    for (sku, product_class), grp in base.groupby(["sku", "Product_class"], sort=False):
        window = window_by_class[product_class]
        forecast_qty = (
            grp.sort_values("month")["qty"]
            .tail(window)
            .mean()
        )
        forecast_rows.append(
            {
                "sku": sku,
                "Product_class": product_class,
                "forecast_qty": forecast_qty,
            }
        )

    result_order = pd.DataFrame(forecast_rows)

    if safety_stock_df is not None:
        result_order = result_order.merge(
            safety_stock_df[["sku", "safety_stock"]],
            on="sku",
            how="left",
        )
    else:
        result_order["safety_stock"] = 0

    if stock_df is not None:
        stock_cols = set(stock_df.columns)
        if {"sku", "stock_qty"}.issubset(stock_cols):
            stock_data = stock_df[["sku", "stock_qty"]]
        elif {"sku", "qty"}.issubset(stock_cols):
            stock_data = stock_df[["sku", "qty"]].rename(columns={"qty": "stock_qty"})
        else:
            raise ValueError(
                "stock_df должен содержать колонки ['sku', 'stock_qty'] или ['sku', 'qty']"
            )
        result_order = result_order.merge(stock_data, on="sku", how="left")
    else:
        result_order["stock_qty"] = 0

    result_order["safety_stock"] = result_order["safety_stock"].fillna(0)
    result_order["stock_qty"] = result_order["stock_qty"].fillna(0)

    result_order["order_qty"] = (
        result_order["forecast_qty"] - result_order["stock_qty"] + result_order["safety_stock"]
    ).clip(lower=0)

    return result_order


stocks_fixed = stocks.copy()
stocks_fixed = stocks_fixed.rename(columns={"sku": "stock_qty_raw", "qty": "sku"})
stocks_fixed["stock_qty"] = pd.to_numeric(stocks_fixed["stock_qty_raw"], errors="coerce").fillna(0)

order_plan = calculate_order_qty(
    sales_data=sales_data,
    xyz_operational=xyz_operational,
    safety_stock_df=result[["sku", "safety_stock"]],
    stock_df=stocks_fixed[["sku", "stock_qty"]],
)

november_sales = (
    sales_data.assign(month=pd.to_datetime(sales_data["month"], errors="coerce"))
    .loc[lambda df: df["month"] == pd.Timestamp("2025-11-01"), ["sku", "qty"]]
    .groupby("sku", as_index=False)["qty"]
    .sum()
    .rename(columns={"qty": "real_sales"})
)

order_plan = order_plan.merge(november_sales, on="sku", how="left")
order_plan["real_sales"] = order_plan["real_sales"].fillna(0)

order_plan.to_csv(data_path / "order_plan.csv", index=False)
order_plan.head()



,sku,Product_class,forecast_qty,safety_stock,stock_qty,order_qty,real_sales
0,КО01828,X - Стабильный,99.015667,0.000000,40.238000,58.777667,117.507
1,НС07823,Y - Средний,14.807333,8.394414,0.020000,23.181747,15.910
2,ТК07914,Y - Средний,26.387667,9.285622,0.001006,35.672282,39.632
3,ЯП24671,Y - Средний,30.929333,14.282514,0.000000,45.211847,58.066
4,ВО13951,Y - Средний,42.612667,13.532284,0.000844,56.144107,58.934


## Итоговая таблица рекомендаций

Результатом этапа является таблица со следующими полями:
- SKU товара;
- XYZ-класс;
- прогноз спроса;
- текущий остаток;
- стандартное отклонение продаж;
- рассчитанный `safety stock`;
- рекомендуемый объём заказа.
- реальные продажи за этот период (real_sales)

Данная таблица может быть:
- использована напрямую для формирования заказа;
- передана в отчёт для бизнеса;
- дополнена ручными корректировками.


## Анализ результатов и интерпретация

В этом разделе проводится качественный анализ результатов:
- как `safety stock` влияет на итоговый заказ;
- для каких товаров заказ равен нулю и почему;
- различия в логике заказа для X и Y товаров;
- потенциальные риски и ограничения подхода.

Особое внимание уделяется товарам:
- с высокой нестабильностью;
- с малыми объёмами продаж;
- с граничными значениями прогноза и остатков.


In [17]:
order_plan["difference"] = order_plan["forecast_qty"] - order_plan["real_sales"]
order_plan.to_csv(data_path / "order_plan.csv", index=False)
order_plan.head()

,sku,Product_class,forecast_qty,safety_stock,stock_qty,order_qty,real_sales,difference
0,КО01828,X - Стабильный,99.015667,0.000000,40.238000,58.777667,117.507,-18.491333
1,НС07823,Y - Средний,14.807333,8.394414,0.020000,23.181747,15.910,-1.102667
2,ТК07914,Y - Средний,26.387667,9.285622,0.001006,35.672282,39.632,-13.244333
3,ЯП24671,Y - Средний,30.929333,14.282514,0.000000,45.211847,58.066,-27.136667
4,ВО13951,Y - Средний,42.612667,13.532284,0.000844,56.144107,58.934,-16.321333


## Выводы

В рамках данного этапа:
- реализован расчёт `safety stock` на основе фактической нестабильности спроса;
- прогноз спроса преобразован в управленческое решение;
- получена воспроизводимая и объяснимая логика заказа.

Данный этап завершает минимально жизнеспособную версию системы автоматизации заказов.
